# ChagaSight — Final Ensemble Evaluation v3

Computes the **official PhysioNet TPR@5%** (primary metric) plus full thesis-ready
metric suite across the 5-fold ensemble.

## Changes vs evaluation_complete_final (v2)
| # | What | Why |
|---|---|---|
| 1 | Per-dataset AUROC NaN guard | SaMi-Trop is all-positive → sklearn raises; added `if len(unique)==2` check |
| 2 | Added Matthews Correlation Coefficient (MCC) | Better single-number summary for severe imbalance |
| 3 | Added Number-Needed-to-Screen (NNS) | Clinical interpretation for thesis |
| 4 | Added calibration reliability diagram | Required for thesis Chapter 4 discussion |
| 5 | Added probability density histogram by class | Visual separation quality |
| 6 | Added per-fold TPR@5% table with variance | Shows ensemble stability |
| 7 | Added DeLong AUROC CI (bootstrap) | Statistical rigour for thesis results table |
| 8 | Removed `val_score=0.0` read from checkpoint for folds without saved val_score | Was silently reading 0 for some checkpoints |
| 9 | All plots saved to `checkpoints/thesis_figures/` at 300 dpi | Ready for LaTeX inclusion |


## Cell 1 — Imports

In [ ]:
import sys, warnings
from pathlib import Path
import torch
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.metrics import (
    roc_curve, precision_recall_curve, confusion_matrix,
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, matthews_corrcoef,
)
warnings.filterwarnings('ignore', category=UserWarning)

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Official PhysioNet metric
helper_path = project_root / 'external' / 'official_2025'
if str(helper_path) not in sys.path:
    sys.path.insert(0, str(helper_path))

OFFICIAL = False
try:
    from helper_code import compute_challenge_score, compute_auc
    OFFICIAL = True
    print(f'Official metric: ON  (helper_code.py found)')
except ImportError:
    print('Official metric: OFF  (helper_code.py not found — using sklearn approx)')

from src.models.hybrid_model import HybridChagasModel
from src.training.dataset import create_dataloaders

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')


## Cell 2 — Configuration

In [ ]:
CHECKPOINT_DIR = project_root / 'checkpoints'
DATA_DIR       = project_root / 'data' / 'processed'
METADATA_CSV   = DATA_DIR / 'metadata' / 'combined_5fold.csv'
IMAGES_DIR     = DATA_DIR / '2d_images'
SIGNALS_DIR    = DATA_DIR / '1d_signals_100hz'
FIGURES_DIR    = CHECKPOINT_DIR / 'thesis_figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

EVAL_CKPT_DIR  = CHECKPOINT_DIR / 'evaluation_checkpoints'
EVAL_CKPT_DIR.mkdir(exist_ok=True)

INFERENCE_BATCH = 32   # safe for 6 GB GPU during inference (no grad)
N_PERMS_FINAL   = 10000
N_BOOTSTRAP     = 1000  # bootstrap iterations for confidence intervals
SEED            = 12345

# Verify fold checkpoints
fold_ckpts = []
for f in range(5):
    p = CHECKPOINT_DIR / f'fold{f}_best.pt'
    assert p.exists(), f'Missing {p} — train fold {f} first'
    fold_ckpts.append(p)
    mb = p.stat().st_size / 1e6
    print(f'fold{f}_best.pt  {mb:.0f} MB')


## Cell 3 — Load All 5 Fold Models

In [ ]:
models = []
fold_val_scores = []

for fold, ckpt_path in enumerate(fold_ckpts):
    m = HybridChagasModel(
        img_size=(24, 2048), patch_size_2d=(8, 64),
        num_leads=12, seq_len_1d=1000, patch_size_1d=50,
        embed_dim=768, depth=12, num_heads=12,
        use_aol=True, use_demographics=True,
    )
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    m.load_state_dict(ckpt['model_state_dict'])
    m.to(device).eval()
    models.append(m)
    vs = ckpt.get('val_score', None)
    fold_val_scores.append(vs)
    score_str = f'{vs:.4f}' if vs is not None else 'n/a (not saved in ckpt)'
    print(f'Fold {fold}: TPR@5%={score_str}')

total_params = sum(p.numel() for p in models[0].parameters())
print(f'\nArchitecture: HybridChagasModel  |  Params: {total_params:,}')
valid_scores = [s for s in fold_val_scores if s is not None]
if valid_scores:
    print(f'Fold scores: mean={np.mean(valid_scores):.4f}  std={np.std(valid_scores):.4f}')


## Cell 4 — Ensemble Inference (batch-level checkpointing)

In [ ]:
import time, json

all_probs, all_labels, all_ids, all_datasets, all_folds = [], [], [], [], []

total_start = time.time()

for fold in range(5):
    fold_complete = EVAL_CKPT_DIR / f'fold{fold}_complete.npz'

    # Resume from completed fold
    if fold_complete.exists():
        d = np.load(fold_complete, allow_pickle=True)
        all_probs.extend(d['probs'].tolist())
        all_labels.extend(d['labels'].tolist())
        all_ids.extend(d['ids'].tolist())
        all_datasets.extend(d['datasets'].tolist())
        all_folds.extend([fold] * len(d['labels']))
        print(f'Fold {fold}: loaded from cache  ({len(d["labels"]):,} samples)')
        continue

    print(f'Fold {fold}: running inference ...')
    fold_start = time.time()

    _, val_loader = create_dataloaders(
        metadata_csv=str(METADATA_CSV),
        images_dir=str(IMAGES_DIR),
        signals_dir=str(SIGNALS_DIR),
        fold=fold,
        batch_size=INFERENCE_BATCH,
        num_workers=0,
        use_weighted_sampling=False,
        augment_train=False,
    )

    partial = EVAL_CKPT_DIR / f'fold{fold}_partial.npz'
    start_batch = 0
    fp, fl, fi, fd = [], [], [], []

    if partial.exists():
        d = np.load(partial, allow_pickle=True)
        fp, fl = d['probs'].tolist(), d['labels'].tolist()
        fi, fd = d['ids'].tolist(), d['datasets'].tolist()
        start_batch = int(d['last_batch']) + 1
        print(f'  Resuming from batch {start_batch}')

    with torch.no_grad():
        for bi, batch in enumerate(tqdm(val_loader, desc=f'Fold {fold}', leave=False)):
            if bi < start_batch:
                continue
            imgs    = batch['image'].to(device, non_blocking=True)
            sigs    = batch['signal'].to(device, non_blocking=True)
            ages    = batch['age'].to(device, non_blocking=True)
            sexes   = batch['sex'].to(device, non_blocking=True)
            hlabels = batch['hard_label'].numpy()

            preds = []
            for m in models:
                out = m(imgs, sigs, ages, sexes)
                preds.append(torch.sigmoid(out['logits']).cpu().numpy())
            ensemble = np.mean(np.stack(preds), axis=0)

            fp.extend(ensemble.tolist())
            fl.extend(hlabels.tolist())
            fi.extend(batch['id'])
            fd.extend(batch['dataset'])

            if (bi + 1) % 50 == 0:
                np.savez(partial, probs=np.array(fp), labels=np.array(fl),
                         ids=fi, datasets=fd, last_batch=bi)

    fold_probs  = np.array(fp)
    fold_labels = np.array(fl)

    np.savez(fold_complete, probs=fold_probs, labels=fold_labels, ids=fi, datasets=fd)
    if partial.exists():
        partial.unlink()

    all_probs.extend(fold_probs.tolist())
    all_labels.extend(fold_labels.tolist())
    all_ids.extend(fi)
    all_datasets.extend(fd)
    all_folds.extend([fold] * len(fold_labels))

    t = time.time() - fold_start
    print(f'  {len(fold_labels):,} samples | {int(fold_labels.sum())} pos | {t/60:.1f} min')
    if device == 'cuda':
        torch.cuda.empty_cache()

all_probs   = np.array(all_probs)
all_labels  = np.array(all_labels)
all_folds_arr = np.array(all_folds)

total_t = time.time() - total_start
print(f'\nInference complete: {len(all_labels):,} samples | {int(all_labels.sum())} pos '
      f'({100*all_labels.mean():.2f}%) | {total_t/60:.1f} min')


## Cell 5 — Primary Metrics (Official PhysioNet)

In [ ]:
np.random.seed(SEED)

# ── TPR@5% ─────────────────────────────────────────────────────────────────
if OFFICIAL:
    tpr_5pct = float(compute_challenge_score(
        all_labels.astype(np.float64), all_probs.astype(np.float64),
        fraction_capacity=0.05, num_permutations=N_PERMS_FINAL, seed=SEED,
    ))
    auroc_off, auprc_off = compute_auc(all_labels, all_probs)
    auroc = float(auroc_off)
    auprc = float(auprc_off)
else:
    fpr_, tpr_, _ = roc_curve(all_labels, all_probs)
    idx = np.where(fpr_ <= 0.05)[0]
    tpr_5pct = float(tpr_[idx[-1]]) if len(idx) > 0 else 0.0
    auroc = float(roc_auc_score(all_labels, all_probs))
    auprc = float(average_precision_score(all_labels, all_probs))

print(f'TPR@5%:  {tpr_5pct:.4f}  [{"OFFICIAL" if OFFICIAL else "APPROX"}  {N_PERMS_FINAL} perms]')
print(f'AUROC:   {auroc:.4f}')
print(f'AUPRC:   {auprc:.4f}')

# Benchmarks
for label, val in [('Random baseline', 0.05), ('PhysioNet target', 0.420),
                   ('Top team (Van Santvliet)', 0.445), ('SOTA (CV)', 0.490)]:
    diff = tpr_5pct - val
    mark = '↑' if diff >= 0 else '↓'
    print(f'{label:<30} {val:.3f}   {mark}{abs(diff):.4f}')

# Clinical
N = len(all_labels)
n_pos = int(all_labels.sum())
capacity = int(0.05 * N)
found    = int(tpr_5pct * n_pos)
nns      = round(capacity / found, 1) if found > 0 else float('inf')
print(f'\nCapacity (5%): {capacity:,} patients | Cases found: {found}/{n_pos} | NNS: {nns}')
print(f'Random would find: {max(1, int(0.05 * n_pos))} cases | Improvement: {found / max(1, int(0.05 * n_pos)):.1f}×')


## Cell 6 — Threshold-Based Metrics

In [ ]:
fpr_arr, tpr_arr, roc_thr = roc_curve(all_labels, all_probs)
prec_arr, rec_arr, pr_thr  = precision_recall_curve(all_labels, all_probs)

# Optimal Youden J
j_idx     = np.argmax(tpr_arr - fpr_arr)
thr_youden = float(roc_thr[j_idx])

# Optimal F1
f1_vals    = 2 * prec_arr[:-1] * rec_arr[:-1] / (prec_arr[:-1] + rec_arr[:-1] + 1e-9)
f1_idx     = np.argmax(f1_vals)
thr_f1     = float(pr_thr[f1_idx])

results = {}
for name, thr in [('default_0.5', 0.5), ('youden_j', thr_youden), ('optimal_f1', thr_f1)]:
    pred = (all_probs >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(all_labels, pred).ravel()
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    npv  = tn / (tn + fn) if (tn + fn) > 0 else 0.0
    mcc  = matthews_corrcoef(all_labels, pred)
    results[name] = dict(
        threshold   = round(thr, 4),
        TP=int(tp), TN=int(tn), FP=int(fp), FN=int(fn),
        sensitivity = round(recall_score(all_labels, pred), 4),
        specificity = round(spec, 4),
        precision   = round(precision_score(all_labels, pred, zero_division=0), 4),
        npv         = round(npv, 4),
        f1          = round(f1_score(all_labels, pred, zero_division=0), 4),
        mcc         = round(float(mcc), 4),
        accuracy    = round(float(accuracy_score(all_labels, pred)), 4),
    )

df_thr = pd.DataFrame(results).T
print('Threshold analysis:')
print(df_thr[['threshold','sensitivity','specificity','precision','npv','f1','mcc','accuracy']].to_string())

# Primary threshold = Youden J (best for screening)
primary = results['youden_j']
print(f'\nPrimary threshold (Youden J = {primary["threshold"]}):')
print(f'  Sensitivity (recall): {primary["sensitivity"]:.4f}')
print(f'  Specificity:          {primary["specificity"]:.4f}')
print(f'  Precision (PPV):      {primary["precision"]:.4f}')
print(f'  NPV:                  {primary["npv"]:.4f}')
print(f'  F1:                   {primary["f1"]:.4f}')
print(f'  MCC:                  {primary["mcc"]:.4f}')
print(f'  Confusion matrix: TP={primary["TP"]} TN={primary["TN"]} FP={primary["FP"]} FN={primary["FN"]}')


## Cell 7 — Bootstrap Confidence Intervals

In [ ]:
np.random.seed(SEED)
n = len(all_labels)
bt_tpr, bt_auroc, bt_auprc = [], [], []

print(f'Bootstrapping {N_BOOTSTRAP} samples ...')
for _ in tqdm(range(N_BOOTSTRAP), leave=False):
    idx = np.random.choice(n, n, replace=True)
    lbl = all_labels[idx]
    prb = all_probs[idx]
    if lbl.sum() < 2 or (lbl == 0).sum() < 2:
        continue
    # fast approx TPR@5% (no permutation loop in bootstrap)
    fpr_b, tpr_b, _ = roc_curve(lbl, prb)
    i5 = np.where(fpr_b <= 0.05)[0]
    bt_tpr.append(float(tpr_b[i5[-1]]) if len(i5) > 0 else 0.0)
    bt_auroc.append(float(roc_auc_score(lbl, prb)))
    bt_auprc.append(float(average_precision_score(lbl, prb)))

def ci95(arr):
    a = np.array(arr)
    return np.percentile(a, 2.5), np.percentile(a, 97.5)

tpr_lo,   tpr_hi   = ci95(bt_tpr)
auroc_lo, auroc_hi = ci95(bt_auroc)
auprc_lo, auprc_hi = ci95(bt_auprc)

print(f'\n95% bootstrap CI (n={N_BOOTSTRAP} resamples):')
print(f'  TPR@5%: {tpr_5pct:.4f}  [{tpr_lo:.4f}, {tpr_hi:.4f}]  (approx; permutation CI)')
print(f'  AUROC:  {auroc:.4f}  [{auroc_lo:.4f}, {auroc_hi:.4f}]')
print(f'  AUPRC:  {auprc:.4f}  [{auprc_lo:.4f}, {auprc_hi:.4f}]')
print()
print('Note: TPR@5% CI uses simplified (ROC-curve) bootstrap, not permutation-based.')
print('Official permutation-based CIs are not standard; report as-is in thesis.')


## Cell 8 — Per-Dataset Analysis

In [ ]:
ds_rows = []
for ds_name in ['ptbxl', 'samitrop', 'code15']:
    mask = np.array([d == ds_name for d in all_datasets])
    if not mask.any():
        continue
    dl, dp = all_labels[mask], all_probs[mask]
    n_pos_ds = int(dl.sum())
    row = dict(dataset=ds_name.upper(), n_total=int(mask.sum()), n_pos=n_pos_ds)

    if len(np.unique(dl)) < 2:
        # Single class (e.g. SaMi-Trop all positive, PTB-XL all negative)
        row.update(tpr_5pct='n/a', auroc='n/a', auprc='n/a', note='single class')
    else:
        if OFFICIAL:
            ds_tpr = float(compute_challenge_score(
                dl.astype(np.float64), dp.astype(np.float64),
                fraction_capacity=0.05, num_permutations=5000, seed=SEED,
            ))
            ds_auroc, ds_auprc = compute_auc(dl, dp)
        else:
            fpr_d, tpr_d, _ = roc_curve(dl, dp)
            i5d = np.where(fpr_d <= 0.05)[0]
            ds_tpr   = float(tpr_d[i5d[-1]]) if len(i5d) > 0 else 0.0
            ds_auroc = float(roc_auc_score(dl, dp))
            ds_auprc = float(average_precision_score(dl, dp))
        row.update(tpr_5pct=round(ds_tpr, 4),
                   auroc=round(float(ds_auroc), 4),
                   auprc=round(float(ds_auprc), 4),
                   note='')

    ds_rows.append(row)

df_ds = pd.DataFrame(ds_rows)
df_ds.to_csv(CHECKPOINT_DIR / 'per_dataset_metrics.csv', index=False)
print(df_ds.to_string(index=False))


## Cell 9 — Per-Fold Performance Table

In [ ]:
fold_rows = []
for fold in range(5):
    mask = all_folds_arr == fold
    fl, fp2 = all_labels[mask], all_probs[mask]
    if len(np.unique(fl)) < 2:
        fold_rows.append(dict(fold=fold, tpr_5pct='n/a', auroc='n/a', auprc='n/a'))
        continue
    if OFFICIAL:
        ft = float(compute_challenge_score(
            fl.astype(np.float64), fp2.astype(np.float64),
            fraction_capacity=0.05, num_permutations=5000, seed=SEED,
        ))
        fa, fp3 = compute_auc(fl, fp2)
    else:
        fpr_f, tpr_f, _ = roc_curve(fl, fp2)
        i5f = np.where(fpr_f <= 0.05)[0]
        ft  = float(tpr_f[i5f[-1]]) if len(i5f) > 0 else 0.0
        fa  = float(roc_auc_score(fl, fp2))
        fp3 = float(average_precision_score(fl, fp2))
    fold_rows.append(dict(fold=fold,
                          tpr_5pct=round(ft, 4),
                          auroc=round(float(fa), 4),
                          auprc=round(float(fp3), 4),
                          n_pos=int(fl.sum()), n_total=int(mask.sum())))

# Add ensemble row
fold_rows.append(dict(fold='Ensemble', tpr_5pct=round(tpr_5pct, 4),
                      auroc=round(auroc, 4), auprc=round(auprc, 4),
                      n_pos=int(all_labels.sum()), n_total=len(all_labels)))

df_folds = pd.DataFrame(fold_rows)
df_folds.to_csv(CHECKPOINT_DIR / 'per_fold_metrics.csv', index=False)
print(df_folds.to_string(index=False))

numeric_tpr = [r['tpr_5pct'] for r in fold_rows[:5] if isinstance(r['tpr_5pct'], float)]
if len(numeric_tpr) == 5:
    print(f'\nFold mean ± std: {np.mean(numeric_tpr):.4f} ± {np.std(numeric_tpr):.4f}')
    print(f'Ensemble gain over mean: +{tpr_5pct - np.mean(numeric_tpr):.4f}')


## Cell 10 — Thesis Figures (300 dpi, FIGURES_DIR)

In [ ]:
def save_fig(name):
    path = FIGURES_DIR / name
    plt.savefig(path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f'Saved: {path.name}')

plt.rcParams.update({'font.size': 11, 'axes.grid': True,
                     'grid.alpha': 0.3, 'axes.spines.top': False,
                     'axes.spines.right': False})

# ── Figure 1: ROC curve ────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(fpr_arr, tpr_arr, lw=2.5, color='#2E86AB', label=f'ChagaSight (AUC = {auroc:.3f})')
ax.plot([0, 1], [0, 1], 'k--', lw=1.2, alpha=0.5, label='Random classifier')
i5 = np.argmin(np.abs(fpr_arr - 0.05))
ax.plot(fpr_arr[i5], tpr_arr[i5], 'ro', ms=10, zorder=5,
        label=f'5% FPR  TPR = {tpr_arr[i5]:.3f}')
ax.axvline(0.05, color='grey', ls=':', lw=1, alpha=0.6)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('Receiver Operating Characteristic')
ax.legend(loc='lower right')
ax.set_xlim(-0.02, 1.02); ax.set_ylim(-0.02, 1.02)
plt.tight_layout()
save_fig('fig_roc_curve.png')

# ── Figure 2: Precision-Recall curve ──────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(rec_arr, prec_arr, lw=2.5, color='#A23B72', label=f'ChagaSight (AP = {auprc:.3f})')
ax.axhline(all_labels.mean(), color='k', ls='--', lw=1.2, alpha=0.5,
           label=f'Random ({all_labels.mean():.3f})')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curve')
ax.legend(loc='upper right')
ax.set_xlim(-0.02, 1.02); ax.set_ylim(-0.02, 1.02)
plt.tight_layout()
save_fig('fig_pr_curve.png')

# ── Figure 3: Confusion matrix ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 5))
thr = primary['threshold']
pred = (all_probs >= thr).astype(int)
cm = confusion_matrix(all_labels, pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Predicted Neg', 'Predicted Pos'],
            yticklabels=['True Neg', 'True Pos'],
            annot_kws={'size': 14, 'weight': 'bold'}, ax=ax,
            cbar_kws={'label': 'Count'})
total_cm = cm.sum()
for i in range(2):
    for j in range(2):
        ax.text(j + 0.5, i + 0.72, f'({100*cm[i,j]/total_cm:.1f}%)',
                ha='center', va='center', fontsize=10, color='dimgrey')
ax.set_title(f'Confusion Matrix  (threshold = {thr:.4f}, Youden J)')
plt.tight_layout()
save_fig('fig_confusion_matrix.png')

# ── Figure 4: Probability histogram by class ────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(all_probs[all_labels == 0], bins=60, alpha=0.6, color='steelblue',
        density=True, label='Negative (n={:,})'.format(int((all_labels == 0).sum())))
ax.hist(all_probs[all_labels == 1], bins=60, alpha=0.6, color='crimson',
        density=True, label='Positive (n={:,})'.format(int(all_labels.sum())))
ax.axvline(thr, color='k', ls='--', lw=1.5, label=f'Threshold {thr:.3f}')
ax.set_xlabel('Predicted Probability')
ax.set_ylabel('Density')
ax.set_title('Predicted Probability Distribution by Class')
ax.legend()
plt.tight_layout()
save_fig('fig_prob_histogram.png')

# ── Figure 5: Calibration reliability diagram ──────────────────────────────
n_bins = 10
bin_edges = np.linspace(0, 1, n_bins + 1)
bin_mids  = (bin_edges[:-1] + bin_edges[1:]) / 2
frac_pos  = np.zeros(n_bins)
mean_pred = np.zeros(n_bins)
bin_counts = np.zeros(n_bins)
for i in range(n_bins):
    mask_b = (all_probs >= bin_edges[i]) & (all_probs < bin_edges[i+1])
    if i == n_bins - 1:
        mask_b = (all_probs >= bin_edges[i]) & (all_probs <= bin_edges[i+1])
    if mask_b.sum() > 0:
        frac_pos[i]   = all_labels[mask_b].mean()
        mean_pred[i]  = all_probs[mask_b].mean()
        bin_counts[i] = mask_b.sum()

fig, (ax_cal, ax_hist) = plt.subplots(2, 1, figsize=(7, 8),
                                       gridspec_kw={'height_ratios': [3, 1]})
ax_cal.plot([0, 1], [0, 1], 'k--', lw=1.2, label='Perfect calibration')
valid = bin_counts > 0
ax_cal.plot(mean_pred[valid], frac_pos[valid], 'o-', lw=2, ms=7,
            color='#E07B39', label='ChagaSight ensemble')
ax_cal.set_xlabel('Mean Predicted Probability')
ax_cal.set_ylabel('Fraction of Positives')
ax_cal.set_title('Calibration Reliability Diagram')
ax_cal.legend()
ax_cal.set_xlim(-0.02, 1.02); ax_cal.set_ylim(-0.02, 1.02)

ax_hist.bar(bin_mids, bin_counts, width=0.08, color='steelblue', alpha=0.7)
ax_hist.set_xlabel('Predicted Probability')
ax_hist.set_ylabel('Count')
ax_hist.set_title('Prediction Histogram')
plt.tight_layout()
save_fig('fig_calibration.png')

# ── Figure 6: Per-fold performance bar ────────────────────────────────────
fold_tpr_vals = [r['tpr_5pct'] if isinstance(r['tpr_5pct'], float) else 0
                 for r in fold_rows]
fold_labels_plot = [f'Fold {r["fold"]}' if r['fold'] != 'Ensemble' else 'Ensemble'
                    for r in fold_rows]
colors = ['#4472C4'] * 5 + ['#ED7D31']

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(range(len(fold_labels_plot)), fold_tpr_vals, color=colors,
              edgecolor='black', linewidth=0.7, alpha=0.85)
ax.axhline(0.42,  color='green',  ls='--', lw=1.5, alpha=0.8, label='Target 0.420')
ax.axhline(0.445, color='red',    ls='--', lw=1.5, alpha=0.8, label='Top team 0.445')
ax.axhline(0.490, color='purple', ls=':',  lw=1.5, alpha=0.8, label='SOTA (CV) 0.490')
for bar, val in zip(bars, fold_tpr_vals):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
            f'{val:.3f}', ha='center', va='bottom', fontsize=10)
ax.set_xticks(range(len(fold_labels_plot)))
ax.set_xticklabels(fold_labels_plot)
ax.set_ylabel('TPR @ 5% FPR')
ax.set_title('Per-Fold and Ensemble Performance (TPR@5%)')
ax.legend(fontsize=9)
ax.set_ylim(0, min(1.0, max(fold_tpr_vals) * 1.18))
plt.tight_layout()
save_fig('fig_per_fold_bar.png')

print(f'\nAll figures saved to: {FIGURES_DIR}')


## Cell 11 — Summary Table (Thesis-Ready)

In [ ]:
summary = {
    'TPR @ 5% FPR (primary)':      f'{tpr_5pct:.4f}  [{tpr_lo:.4f}–{tpr_hi:.4f}]',
    'AUROC':                        f'{auroc:.4f}  [{auroc_lo:.4f}–{auroc_hi:.4f}]',
    'AUPRC':                        f'{auprc:.4f}  [{auprc_lo:.4f}–{auprc_hi:.4f}]',
    'Sensitivity (Youden J thr)':   f'{primary["sensitivity"]:.4f}',
    'Specificity (Youden J thr)':   f'{primary["specificity"]:.4f}',
    'Precision / PPV':              f'{primary["precision"]:.4f}',
    'NPV':                          f'{primary["npv"]:.4f}',
    'F1 Score':                     f'{primary["f1"]:.4f}',
    'MCC':                          f'{primary["mcc"]:.4f}',
    'Accuracy':                     f'{primary["accuracy"]:.4f}',
    'Threshold (Youden J)':         f'{primary["threshold"]:.4f}',
    'TP / TN / FP / FN':           f'{primary["TP"]} / {primary["TN"]:,} / {primary["FP"]:,} / {primary["FN"]}',
    'Total samples':                f'{len(all_labels):,}',
    'Positive samples':             f'{int(all_labels.sum()):,}  ({100*all_labels.mean():.2f}%)',
    'Ensemble size':                '5 folds',
    'Params per model':             f'{total_params:,}',
    'Metric method':                'OFFICIAL PhysioNet' if OFFICIAL else 'sklearn approx',
    'Bootstrap CI resamples':       f'{N_BOOTSTRAP}',
}

df_summary = pd.DataFrame.from_dict(summary, orient='index', columns=['Value'])
df_summary.index.name = 'Metric'
print(df_summary.to_string())

# Save all outputs
df_summary.to_csv(CHECKPOINT_DIR / 'ensemble_summary.csv')
df_thr.to_csv(CHECKPOINT_DIR / 'threshold_comparison.csv')

pred_df = pd.DataFrame({
    'id': all_ids, 'fold': all_folds_arr, 'dataset': all_datasets,
    'true_label': all_labels, 'predicted_probability': all_probs,
    'predicted_class': (all_probs >= primary['threshold']).astype(int),
})
pred_df.to_csv(CHECKPOINT_DIR / 'ensemble_predictions.csv', index=False)

print(f'\nFiles saved:')
for f in ['ensemble_summary.csv', 'threshold_comparison.csv',
          'per_dataset_metrics.csv', 'per_fold_metrics.csv',
          'ensemble_predictions.csv']:
    print(f'  {f}')
print(f'  thesis_figures/  ({len(list(FIGURES_DIR.glob("*.png")))} PNG files)')


## Cell 12 — Package Ensemble Model

In [ ]:
ensemble_pkg = {
    'model_config': dict(
        img_size=(24, 2048), patch_size_2d=(8, 64),
        num_leads=12, seq_len_1d=1000, patch_size_1d=50,
        embed_dim=768, depth=12, num_heads=12,
        use_aol=True, use_demographics=True,
    ),
    'fold_models': [],
    'ensemble_metrics': {
        'tpr_5pct': tpr_5pct, 'auroc': auroc, 'auprc': auprc,
        'threshold': primary['threshold'],
        'n_total': len(all_labels), 'n_positive': int(all_labels.sum()),
    },
    'fold_scores': fold_val_scores,
}
for fold, ckpt_path in enumerate(fold_ckpts):
    ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    ensemble_pkg['fold_models'].append({
        'fold': fold,
        'model_state_dict': ckpt['model_state_dict'],
        'val_score': ckpt.get('val_score', None),
    })

out_path = CHECKPOINT_DIR / 'FINAL_ENSEMBLE_MODEL.pt'
torch.save(ensemble_pkg, out_path)
mb = out_path.stat().st_size / 1e6
print(f'Saved: FINAL_ENSEMBLE_MODEL.pt  ({mb:.0f} MB)')
print('Contains: 5 fold weights + metrics + config')
